# ASR Fine-Tuning Experiment: Whisper on LibriSpeech

This notebook adds the ASR phase of the spoken narrative evaluation pipeline.

The goal is not to train an ASR system from scratch. Instead, we fine-tune a pretrained Whisper model on a small, controlled subset of LibriSpeech and compare transcription quality before and after fine-tuning using WER and CER.

Pipeline position:

`voice cleanliness gate -> ASR transcription -> coherence checking`


## Experiment Design

- **Base model:** `openai/whisper-tiny` by default, because it is small enough for local experimentation.
- **Dataset:** LibriSpeech ASR, clean configuration.
- **Training strategy:** small streamed subset to avoid downloading the full dataset.
- **Evaluation metrics:** Word Error Rate (WER) and Character Error Rate (CER).
- **Model selection:** best checkpoint is selected using validation WER, lower is better.

You can increase the number of samples after the notebook runs successfully once.


In [1]:
from pathlib import Path
import io
import json
import os
import random
import re
import time
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import numpy as np
import pandas as pd

# Keep this ASR notebook PyTorch-only. The local TensorFlow install is not needed here.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import soundfile as sf

from datasets import Audio, Dataset, load_dataset
from jiwer import cer as jiwer_cer, wer as jiwer_wer

# Compatibility patch for a mismatched Anaconda Hugging Face install.
# Some local installs expose huggingface_hub 1.x but miss KernelInfo in hf_api,
# which prevents transformers from importing. This provides the missing symbol.
try:
    import huggingface_hub.hf_api as _hf_api
    if not hasattr(_hf_api, "KernelInfo"):
        class KernelInfo:
            pass
        _hf_api.KernelInfo = KernelInfo
except Exception:
    pass

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUTPUT_DIR = Path("outputs/asr_whisper_librispeech")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


c:\Users\ah625\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\ah625\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Device: cuda


In [2]:
MODEL_NAME = "openai/whisper-base"
DATASET_NAME = "openslr/librispeech_asr"
DATASET_CONFIG = "clean"

# Streaming prevents the full dataset from being downloaded.
USE_STREAMING = True

# Start small. Increase these only after the full notebook works once.
MAX_TRAIN_SAMPLES = 1000
MAX_VAL_SAMPLES = 200
MAX_TEST_SAMPLES = 200

# LibriSpeech clean split names on Hugging Face.
TRAIN_SPLIT = "train.100"
VAL_SPLIT = "validation"
TEST_SPLIT = "test"

SAMPLE_RATE = 16000
MAX_LABEL_LENGTH = 448

BATCH_SIZE = 8
GRAD_ACCUMULATION = 2
LEARNING_RATE = 1e-5
NUM_EPOCHS = 3
WARMUP_STEPS = 25
EVAL_STEPS = 50
SAVE_STEPS = 50

run_config = {
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "dataset_config": DATASET_CONFIG,
    "use_streaming": USE_STREAMING,
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "max_val_samples": MAX_VAL_SAMPLES,
    "max_test_samples": MAX_TEST_SAMPLES,
    "train_split": TRAIN_SPLIT,
    "val_split": VAL_SPLIT,
    "test_split": TEST_SPLIT,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUMULATION,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
}

with open(OUTPUT_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

run_config


{'model_name': 'openai/whisper-base',
 'dataset_name': 'openslr/librispeech_asr',
 'dataset_config': 'clean',
 'use_streaming': True,
 'max_train_samples': 1000,
 'max_val_samples': 200,
 'max_test_samples': 200,
 'train_split': 'train.100',
 'val_split': 'validation',
 'test_split': 'test',
 'sample_rate': 16000,
 'batch_size': 8,
 'gradient_accumulation': 2,
 'learning_rate': 1e-05,
 'num_epochs': 3}

## Load a Small ASR Dataset Subset

This cell uses streaming and then materializes only a small number of examples into memory. This avoids downloading the full LibriSpeech dataset.


In [3]:
def load_streamed_subset(split: str, max_samples: int, seed: int = SEED) -> Dataset:
    stream = load_dataset(
        DATASET_NAME,
        DATASET_CONFIG,
        split=split,
        streaming=True,
    )
    stream = stream.cast_column("audio", Audio(decode=False))
    stream = stream.shuffle(seed=seed, buffer_size=1000)
    rows = list(stream.take(max_samples))
    dataset = Dataset.from_list(rows)
    dataset = dataset.cast_column("audio", Audio(decode=False))
    return dataset


def load_regular_subset(split: str, max_samples: int, seed: int = SEED) -> Dataset:
    dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=split)
    dataset = dataset.cast_column("audio", Audio(decode=False))
    if max_samples and max_samples < len(dataset):
        dataset = dataset.shuffle(seed=seed).select(range(max_samples))
    dataset = dataset.cast_column("audio", Audio(decode=False))
    return dataset

def decode_audio(audio_value) -> dict:
    if isinstance(audio_value, dict) and audio_value.get("array") is not None:
        array = np.asarray(audio_value["array"], dtype=np.float32)
        sr = int(audio_value.get("sampling_rate", SAMPLE_RATE))
    elif isinstance(audio_value, dict) and audio_value.get("bytes") is not None:
        array, sr = sf.read(io.BytesIO(audio_value["bytes"]), dtype="float32")
    elif isinstance(audio_value, dict) and audio_value.get("path") is not None:
        array, sr = sf.read(audio_value["path"], dtype="float32")
    else:
        raise ValueError(f"Unsupported audio value: {type(audio_value)}")

    if array.ndim > 1:
        array = array.mean(axis=1)
    if sr != SAMPLE_RATE:
        import librosa
        array = librosa.resample(array, orig_sr=sr, target_sr=SAMPLE_RATE)
        sr = SAMPLE_RATE
    return {"array": array.astype(np.float32), "sampling_rate": sr}


loader = load_streamed_subset if USE_STREAMING else load_regular_subset

train_ds_raw = loader(TRAIN_SPLIT, MAX_TRAIN_SAMPLES)
val_ds_raw = loader(VAL_SPLIT, MAX_VAL_SAMPLES)
test_ds_raw = loader(TEST_SPLIT, MAX_TEST_SAMPLES)

print(train_ds_raw)
print(val_ds_raw)
print(test_ds_raw)
print(train_ds_raw[0].keys())
print("Example transcript:", train_ds_raw[0]["text"])


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Dataset({
    features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
    num_rows: 1000
})
Dataset({
    features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
    num_rows: 200
})
Dataset({
    features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
    num_rows: 200
})
dict_keys(['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'])
Example transcript: MULLINS AND BAGSHAW AND JUDGE PEPPERLEIGH AND THE REST ARE IT IS TRUE PERSONAL FRIENDS OF MINE BUT I HAVE KNOWN THEM IN SUCH A VARIETY OF FORMS WITH SUCH ALTERNATIONS OF TALL AND SHORT DARK AND FAIR THAT INDIVIDUALLY I SHOULD HAVE MUCH ADO TO KNOW THEM


In [4]:
def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9' ]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

sample_rows = []
for idx in range(min(5, len(train_ds_raw))):
    row = train_ds_raw[idx]
    sample_rows.append({
        "id": row.get("id", idx),
        "duration_seconds": round(len(decode_audio(row["audio"])["array"]) / SAMPLE_RATE, 2),
        "text": row["text"],
        "normalized_text": normalize_text(row["text"]),
    })

pd.DataFrame(sample_rows)


,id,duration_seconds,text,normalized_text
0,460-172357-0021,13.73,MULLINS AND BAGSHAW AND JUDGE PEPPERLEIGH AND ...,mullins and bagshaw and judge pepperleigh and ...
1,4898-20016-0017,13.78,THE MORE HE WAS FILLED FOR ALL THEIR APPEARANC...,the more he was filled for all their appearanc...
2,2518-154826-0010,13.28,IT WAS AFTER NOON FOR THE SUN WAS PAST THE MER...,it was after noon for the sun was past the mer...
3,302-123516-0014,15.27,RATHER THAN BY A WATER HEARTED WEAKLING FROM W...,rather than by a water hearted weakling from w...
4,302-123516-0009,15.64,WHICH SEEMED TO MARK ALL THE GREATER WARRIORS ...,which seemed to mark all the greater warriors ...


## Load Whisper Processor and Model

The processor handles both audio feature extraction and text tokenization. The model is loaded once for baseline evaluation and then fine-tuned.


In [5]:
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="english", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# These settings make fine-tuning behave like direct English transcription.
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.language = "english"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

model.to(DEVICE)
print("Loaded", MODEL_NAME)


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Loaded openai/whisper-base


## Baseline Evaluation Before Fine-Tuning

This evaluates pretrained Whisper directly on the test subset. These numbers are the baseline that the fine-tuned model should be compared against.


In [6]:
def compute_wer(references, predictions):
    return jiwer_wer(references, predictions)


def compute_cer(references, predictions):
    return jiwer_cer(references, predictions)


def transcribe_batch(dataset: Dataset, max_examples: int | None = None) -> pd.DataFrame:
    model.eval()
    rows = []
    n = len(dataset) if max_examples is None else min(len(dataset), max_examples)
    for idx in range(n):
        item = dataset[idx]
        audio = decode_audio(item["audio"])
        input_features = processor(
            audio["array"],
            sampling_rate=audio["sampling_rate"],
            return_tensors="pt",
        ).input_features.to(DEVICE)

        with torch.no_grad():
            predicted_ids = model.generate(input_features, max_new_tokens=128)

        prediction = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        reference = item["text"]
        rows.append({
            "id": item.get("id", idx),
            "reference": reference,
            "prediction": prediction,
            "reference_norm": normalize_text(reference),
            "prediction_norm": normalize_text(prediction),
        })
    return pd.DataFrame(rows)

baseline_predictions = transcribe_batch(test_ds_raw)
baseline_wer = compute_wer(
    references=baseline_predictions["reference_norm"].tolist(),
    predictions=baseline_predictions["prediction_norm"].tolist(),
)
baseline_cer = compute_cer(
    references=baseline_predictions["reference_norm"].tolist(),
    predictions=baseline_predictions["prediction_norm"].tolist(),
)

baseline_metrics = {"stage": "pretrained_baseline", "wer": baseline_wer, "cer": baseline_cer}
pd.DataFrame([baseline_metrics]).to_csv(OUTPUT_DIR / "baseline_metrics.csv", index=False)
baseline_predictions.to_csv(OUTPUT_DIR / "baseline_predictions.csv", index=False)

print(baseline_metrics)
baseline_predictions.head(10)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


{'stage': 'pretrained_baseline', 'wer': 0.0535031847133758, 'cer': 0.022935337766213742}


,id,reference,prediction,reference_norm,prediction_norm
0,1320-122617-0011,THE LODGE IN WHICH UNCAS WAS CONFINED WAS IN T...,The lodge in which Onkus was confined was in ...,the lodge in which uncas was confined was in t...,the lodge in which onkus was confined was in t...
1,1995-1826-0023,GOOBERS DON'T GROW ON THE TOPS OF VINES BUT UN...,"Goobies don't grow on the tops of vines, but ...",goobers don't grow on the tops of vines but un...,goobies don't grow on the tops of vines but on...
2,8230-279154-0037,RECOGNITION IN THIS SENSE DOES NOT NECESSARILY...,Recognition in this sense does not necessaril...,recognition in this sense does not necessarily...,recognition in this sense does not necessarily...
3,7127-75947-0012,INDEED AH,"Indeed, ah.",indeed ah,indeed ah
4,7127-75947-0007,SHE THEN ROSE HUMMING THE AIR TO WHICH SHE WAS...,"She then rose, humming the air to which she w...",she then rose humming the air to which she was...,she then rose humming the air to which she was...
5,1284-1180-0029,SOMETIMES IT IS CALLED A CRAZY QUILT BECAUSE T...,Sometimes it is called a crazy quilt because ...,sometimes it is called a crazy quilt because t...,sometimes it is called a crazy quilt because t...
6,1320-122617-0007,COME COME RETURNED HAWKEYE UNCASING HIS HONEST...,"Com, com, returned Hawkeye, encasing his hone...",come come returned hawkeye uncasing his honest...,com com returned hawkeye encasing his honest c...
7,7176-92135-0036,THE CROWD DRIFTS OFF LEAVING THE HERO AND HERO...,"The crowd drifts off, leaving the hero and he...",the crowd drifts off leaving the hero and hero...,the crowd drifts off leaving the hero and hero...
8,260-123286-0001,THE HORIZON SEEMS EXTREMELY DISTANT,The horizon seems extremely distant.,the horizon seems extremely distant,the horizon seems extremely distant
9,1320-122617-0016,THE CUNNING MAN IS AFRAID THAT HIS BREATH WILL...,The cunning man is afraid that his breath wil...,the cunning man is afraid that his breath will...,the cunning man is afraid that his breath will...


## Prepare Dataset for Fine-Tuning

Whisper expects log-Mel input features and tokenized transcript labels.


In [7]:
def prepare_example(batch):
    audio = decode_audio(batch["audio"])
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_features[0]
    labels = processor.tokenizer(batch["text"]).input_ids
    if len(labels) > MAX_LABEL_LENGTH:
        labels = labels[:MAX_LABEL_LENGTH]
    batch["labels"] = labels
    return batch

train_ds = train_ds_raw.map(prepare_example, remove_columns=train_ds_raw.column_names)
val_ds = val_ds_raw.map(prepare_example, remove_columns=val_ds_raw.column_names)
test_ds = test_ds_raw.map(prepare_example, remove_columns=test_ds_raw.column_names)

print(train_ds)
print(val_ds)
print(test_ds)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['input_features', 'labels'],
    num_rows: 1000
})
Dataset({
    features: ['input_features', 'labels'],
    num_rows: 200
})
Dataset({
    features: ['input_features', 'labels'],
    num_rows: 200
})


In [8]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


In [9]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_norm = [normalize_text(x) for x in pred_str]
    label_norm = [normalize_text(x) for x in label_str]

    wer = compute_wer(references=label_norm, predictions=pred_norm)
    cer = compute_cer(references=label_norm, predictions=pred_norm)
    return {"wer": wer, "cer": cer}


## Fine-Tune Whisper

The best checkpoint is selected using validation WER. Lower WER is better.


In [10]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=NUM_EPOCHS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    generation_max_length=128,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

start = time.time()
train_result = trainer.train()
training_seconds = time.time() - start

trainer.save_model(str(OUTPUT_DIR / "best_model"))
processor.save_pretrained(str(OUTPUT_DIR / "best_model"))

train_metrics = train_result.metrics
train_metrics["training_seconds"] = training_seconds
with open(OUTPUT_DIR / "train_metrics.json", "w", encoding="utf-8") as f:
    json.dump(train_metrics, f, indent=2)

train_metrics


C:\Users\ah625\AppData\Local\Temp\ipykernel_29804\3104864266.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss,Wer,Cer
50,0.480200,0.522903,0.212676,0.082464
100,0.320400,0.445669,0.183214,0.070922
150,0.261300,0.425540,0.176776,0.066657


c:\Users\ah625\anaconda3\Lib\site-packages\transformers\modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


{'train_runtime': 370.0614,
 'train_samples_per_second': 8.107,
 'train_steps_per_second': 0.503,
 'total_flos': 1.919855886336e+17,
 'train_loss': 0.4452812030751218,
 'epoch': 2.96,
 'training_seconds': 370.42824816703796}

## Final Evaluation After Fine-Tuning

This evaluates the selected best checkpoint on the held-out test subset and compares it with the pretrained baseline.


In [11]:
final_eval = trainer.evaluate(test_ds, metric_key_prefix="test")
with open(OUTPUT_DIR / "final_eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump(final_eval, f, indent=2)

comparison = pd.DataFrame([
    {"stage": "pretrained_baseline", "wer": baseline_wer, "cer": baseline_cer},
    {"stage": "fine_tuned", "wer": final_eval["test_wer"], "cer": final_eval["test_cer"]},
])
comparison["wer_percent"] = comparison["wer"] * 100
comparison["cer_percent"] = comparison["cer"] * 100
comparison.to_csv(OUTPUT_DIR / "asr_model_comparison.csv", index=False)
comparison


,stage,wer,cer,wer_percent,cer_percent
0,pretrained_baseline,0.053503,0.022935,5.350318,2.293534
1,fine_tuned,0.179873,0.073046,17.987261,7.304616


In [12]:
# Generate readable final predictions from the fine-tuned model.
fine_tuned_predictions = transcribe_batch(test_ds_raw)
fine_tuned_predictions.to_csv(OUTPUT_DIR / "fine_tuned_predictions.csv", index=False)
fine_tuned_predictions.head(10)


,id,reference,prediction,reference_norm,prediction_norm
0,1320-122617-0011,THE LODGE IN WHICH UNCAS WAS CONFINED WAS IN T...,THE LODGE IN WHICH UNCUS WAS CONFINED WAS IN ...,the lodge in which uncas was confined was in t...,the lodge in which uncus was confined was in t...
1,1995-1826-0023,GOOBERS DON'T GROW ON THE TOPS OF VINES BUT UN...,GOOBAS DON'T GROW ON TO TOP ZEDVICE BUT ON TO...,goobers don't grow on the tops of vines but un...,goobas don't grow on to top zedvice but on to ...
2,8230-279154-0037,RECOGNITION IN THIS SENSE DOES NOT NECESSARILY...,RECOGNITION IN THIS SINCE DOES NOT NEXTCERALL...,recognition in this sense does not necessarily...,recognition in this since does not nextcerally...
3,7127-75947-0012,INDEED AH,INDEED A,indeed ah,indeed a
4,7127-75947-0007,SHE THEN ROSE HUMMING THE AIR TO WHICH SHE WAS...,SHE THEN ROSE HUMMING THE AIR TO WHICH SHE WA...,she then rose humming the air to which she was...,she then rose humming the air to which she was...
5,1284-1180-0029,SOMETIMES IT IS CALLED A CRAZY QUILT BECAUSE T...,SOMETIMES IT IS CALLED A CRAZY QUILT BECAUSE T...,sometimes it is called a crazy quilt because t...,sometimes it is called a crazy quilt because t...
6,1320-122617-0007,COME COME RETURNED HAWKEYE UNCASING HIS HONEST...,COMCOM RETURNED HAKI UNCASING HIS HonEST COUNT...,come come returned hawkeye uncasing his honest...,comcom returned haki uncasing his honest count...
7,7176-92135-0036,THE CROWD DRIFTS OFF LEAVING THE HERO AND HERO...,THE CROWD DRIFTS OFF LEAPING THE HERO AND HER...,the crowd drifts off leaving the hero and hero...,the crowd drifts off leaping the hero and hero...
8,260-123286-0001,THE HORIZON SEEMS EXTREMELY DISTANT,THE HORIZONS SEEMS EXTRAMELY DISTANT,the horizon seems extremely distant,the horizons seems extramely distant
9,1320-122617-0016,THE CUNNING MAN IS AFRAID THAT HIS BREATH WILL...,THE CUNNING MAN IS AFRID THAT HIS BREATH WILL...,the cunning man is afraid that his breath will...,the cunning man is afrid that his breath will ...


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(comparison["stage"], comparison["wer_percent"])
plt.ylabel("WER (%)")
plt.title("ASR Word Error Rate Before and After Fine-Tuning")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "wer_comparison.png", dpi=200)
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(comparison["stage"], comparison["cer_percent"])
plt.ylabel("CER (%)")
plt.title("ASR Character Error Rate Before and After Fine-Tuning")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cer_comparison.png", dpi=200)
plt.show()


C:\Users\ah625\AppData\Local\Temp\ipykernel_29804\2487365037.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ah625\AppData\Local\Temp\ipykernel_29804\2487365037.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Thesis Interpretation Template

The ASR module was evaluated as an intermediate component in the spoken narrative pipeline. A pretrained Whisper model was first tested without fine-tuning to establish a baseline WER and CER. The model was then fine-tuned on a controlled subset of LibriSpeech and evaluated on a held-out test subset. This allows the project to report whether domain-specific ASR fine-tuning improves transcription quality before the text is passed to the coherence checker.

If fine-tuning improves WER/CER, it supports using a locally adapted ASR model. If it does not improve performance, the pretrained Whisper baseline can still be justified as the more reliable transcription component, while the fine-tuning experiment remains academically useful as an ablation study.
